# PowerPoint 기반 RAG 시스템 구축

### 사전 준비

이 노트북을 실행하기 위해 필요한 환경:

- Qdrant 서버 (localhost:6333)
- OpenRouter API key
- .env 파일

- PPT를 PDF로 사전 변환하여 `data/bcg_ppt/` 폴더에 넣어주세요.

`.env` 파일에 `OPENROUTER_API_KEY`를 설정하세요.


> **Note:** 이 노트북은 현재 약 60% 완성된 상태입니다. 문서 파싱, 이미지 추출, VLM 캡셔닝까지 구현되어 있으며, Agentic RAG 검색/생성 부분은 추후 완성 예정입니다.

In [ ]:
import os

from dotenv import load_dotenv
from langchain_openai import ChatOpenAI

load_dotenv()


def create_openrouter_llm(
    model: str = "openai/gpt-4.1-mini",
    temperature: float = 0.3,
    max_tokens: int | None = None,
    **kwargs,
) -> ChatOpenAI:
    """OpenRouter 기반 LLM 생성 헬퍼."""
    api_key = os.getenv("OPENROUTER_API_KEY")
    if not api_key:
        raise RuntimeError("OPENROUTER_API_KEY가 필요합니다.")
    base_url = os.getenv("OPENROUTER_API_BASE") or "https://openrouter.ai/api/v1"
    return ChatOpenAI(
        model=model,
        api_key=api_key,
        base_url=base_url,
        temperature=temperature,
        max_tokens=max_tokens,
        max_retries=3,
        timeout=60,
        **kwargs,
    )


llm = create_openrouter_llm()
print(f"LLM 초기화 완료: {llm.model_name}")

In [ ]:
%pip install pdf2image

In [ ]:
import os
from pathlib import Path

import pymupdf4llm

# PDF 파일들이 있는 폴더 경로
pdf_folder = "data/bcg_ppt"

# 폴더 내의 모든 PDF 파일 찾기
pdf_files = list(Path(pdf_folder).glob("*.pdf"))

# 각 PDF 파일을 마크다운으로 변환
md_texts = {}
for pdf_file in pdf_files:
    print(f"Processing: {pdf_file}")
    md_text = pymupdf4llm.to_markdown(str(pdf_file), page_chunks=True)
    md_texts[pdf_file.name] = md_text

print(f"총 {len(md_texts)}개의 PDF 파일을 처리했습니다.")

### Document 객체 생성

각 PDF 파일의 페이지별 마크다운 텍스트를 LangChain Document 객체로 변환합니다.

이를 통해 벡터 데이터베이스에서 사용할 수 있는 형태로 데이터를 준비합니다.


In [ ]:
from langchain_core.documents import Document

# md_texts의 각 PDF 파일에 대해 Document 객체들을 생성
documents = []
for pdf_name, pages in md_texts.items():
    for page in pages:
        doc = Document(
            page_content=page["text"],
            metadata={"file_name": pdf_name, **page["metadata"]},
        )
        documents.append(doc)

print(f"총 {len(documents)}개의 Document가 생성되었습니다.")

### PDF를 이미지로 변환

각 PDF 파일의 페이지를 PNG 이미지로 변환하여 저장합니다.

PyMuPDF(fitz)를 사용하여 PDF의 각 페이지를 픽스맵으로 변환한 후 이미지 파일로 저장합니다.


In [ ]:
import base64
import time
from pathlib import Path

from dotenv import load_dotenv
from langchain_core.messages import HumanMessage

load_dotenv()
llm = create_openrouter_llm(model="openai/gpt-4o", temperature=0.3)


def create_page_caption(image_path: Path, page_text: str) -> HumanMessage | None:
    """페이지 텍스트를 컨텍스트로 활용한 캡션 생성 메시지"""
    try:
        with open(image_path, "rb") as image_file:
            encoded_image = base64.b64encode(image_file.read()).decode("utf-8")

        prompt = f"""다음은 BCG 컨설팅 리포트의 한 페이지입니다.
        이미지를 분석하여 페이지의 내용을 정확하고 상세하게 텍스트로 변환해주세요.

참고용 추출 텍스트 (순서나 구조가 부정확할 수 있음):
{page_text[:1000]}{"..." if len(page_text) > 1000 else ""}

위 텍스트는 참고용이며, 실제 이미지의 내용과 구조가 다를 수 있습니다.
이미지를 직접 분석하여 다음과 같이 작성해주세요:

1. 페이지의 제목이나 헤더
2. 주요 텍스트 내용 (단락별로 구조화)
3. 차트, 그래프, 표가 있다면 그 내용과 데이터
4. 이미지나 다이어그램이 있다면 설명
5. 페이지 하단의 각주나 출처 정보

모든 텍스트를 읽기 쉽고 구조적으로 정리하여 한글로 작성해주세요."""

        return HumanMessage(
            content=[
                {"type": "text", "text": prompt},
                {"type": "image_url", "image_url": f"data:image/png;base64,{encoded_image}"},
            ]
        )
    except Exception as e:
        print(f"메시지 생성 오류: {e}")
        return None


def pdf_images_with_captions() -> dict[str, str]:
    """각 PDF의 페이지 이미지에 대해 캡셔닝 수행"""
    all_captions = {}

    for pdf_file in pdf_files:
        pdf_name = pdf_file.stem
        image_folder = Path(pdf_folder) / f"{pdf_name}_images"

        if not image_folder.exists():
            print(f"이미지 폴더가 없습니다: {image_folder}")
            continue

        # 해당 PDF의 documents 찾기
        pdf_documents = [doc for doc in documents if doc.metadata["file_name"] == str(pdf_file)]

        # 페이지별 이미지 파일 찾기
        image_files = sorted(image_folder.glob("page_*.png"))

        batch_data = []
        for i, image_file in enumerate(image_files):
            page_num = i  # 0부터 시작하는 페이지 번호

            # 해당 페이지의 텍스트 찾기
            page_text = ""
            if page_num < len(pdf_documents):
                page_text = pdf_documents[page_num].page_content

            batch_data.append((image_file, page_text, f"{pdf_name}_page_{page_num + 1}"))

        batch_size = 10
        for batch_idx in range(0, len(batch_data), batch_size):
            batch = batch_data[batch_idx : batch_idx + batch_size]
            batch_num = batch_idx // batch_size + 1
            total_batches = (len(batch_data) + batch_size - 1) // batch_size

            # 배치용 메시지 준비
            batch_messages = []
            batch_keys = []

            for image_file, page_text, page_key in batch:
                message = create_page_caption(image_file, page_text)
                if message:
                    batch_messages.append([message])
                    batch_keys.append(page_key)

            if batch_messages:
                try:
                    batch_results = llm.batch(batch_messages)

                    for i, result in enumerate(batch_results):
                        if i < len(batch_keys):
                            caption = result.content.strip()
                            all_captions[batch_keys[i]] = caption
                            print(f"    {batch_keys[i]}: {caption[:50]}...")

                except Exception as e:
                    print(f"  배치 처리 오류: {e}")
                    # 개별 처리로 폴백
                    for i, message_list in enumerate(batch_messages):
                        try:
                            result = llm.invoke(message_list)
                            caption = result.content.strip()
                            all_captions[batch_keys[i]] = caption
                            print(f"    {batch_keys[i]}: {caption[:50]}...")
                            time.sleep(0.1)  # 개별 요청 간 짧은 대기
                        except Exception as e:
                            print(f"    {batch_keys[i]}: 캡션 생성 실패")
                            all_captions[batch_keys[i]] = "[캡션 생성 실패]"

            if batch_num < total_batches:
                # Rate Limit
                wait_time = 1.0
                time.sleep(wait_time)

    return all_captions


# 캡셔닝 실행
captions = pdf_images_with_captions()
print(f"총 {len(captions)}개 페이지 캡션 생성됨")

# 결과 확인
for key, caption in list(captions.items())[:3]:
    print(f"\n{key}: {caption}")

### **이미지 캡셔닝을 통한 텍스트 추출**

앞서 생성한 PDF 페이지 이미지들을 VLM 모델을 사용하여 분석하고, 각 페이지의 내용을 상세하게 텍스트로 변환합니다.

- `create_page_caption()`: PDF 페이지 이미지와 기존 추출된 텍스트를 함께 분석하여 정확한 캡션 생성
- `pdf_images_with_captions()`: 모든 PDF 이미지에 대해 캡셔닝을 수행하고 결과를 저장

In [ ]:
for doc in documents[:10]:
    # 문서 메타데이터에서 페이지 번호 추출
    page_num = doc.metadata.get("page")
    source = doc.metadata.get("file_name", "")

    # 이미지 캡션 키 생성 (pdf_images_with_captions에서 사용한 것과 동일)
    # 소스 파일명에서 확장자 제거 후 page 번호 추가
    source_basename = source.split("/")[-1].replace(".pdf", "")
    caption_key = f"{source_basename}_page_{page_num}"
    print(caption_key)

In [ ]:
# 문서 페이지 내용을 캡션으로 업데이트
# PDF 페이지에서 추출된 텍스트 대신 캡션을 사용하여 문서 객체들을 업데이트합니다.
updated_documents = []
for doc in documents:
    # 문서 메타데이터에서 페이지 번호 추출
    page_num = doc.metadata.get("page")
    source = doc.metadata.get("file_name", "")

    # 이미지 캡션 키 생성 (process_pdf_images_with_captions에서 사용한 것과 동일)
    # 소스 파일명에서 확장자 제거 후 page 번호 추가
    source_basename = source.split("/")[-1].replace(".pdf", "")
    caption_key = f"{source_basename}_page_{page_num}"

    # 해당 페이지의 캡션 가져오기
    if caption_key in captions:
        # 새로운 Document 생성 (page_content를 캡션으로 대체)
        updated_doc = Document(
            page_content=captions[caption_key],
            metadata=doc.metadata.copy(),  # 기존 메타데이터 유지
        )
        updated_documents.append(updated_doc)

print(f"\n총 {len(updated_documents)}개 문서에 대해 이미지 캡셔닝 업데이트 완료")

# 업데이트된 문서들 확인
print("\n업데이트된 문서 확인:")
for i, doc in enumerate(updated_documents[:5]):
    print(f"\n문서 {i + 1}:")
    print(f"  메타데이터: {doc.metadata}")
    print(f"  내용 (처음 100자): {doc.page_content}...")

documents = updated_documents

### 벡터 스토어 설정 및 문서 저장

Qdrant를 사용하여 하이브리드 검색(Dense + Sparse) 벡터 스토어를 설정하고, 업데이트된 문서들을 저장합니다.


In [ ]:
from langchain_openai import OpenAIEmbeddings
from langchain_qdrant import QdrantVectorStore, RetrievalMode
from qdrant_client import QdrantClient, models
from qdrant_client.http.models import Distance, SparseVectorParams, VectorParams

# Qdrant 클라이언트 설정
client = QdrantClient(host="localhost", port=6333)
dense_embeddings = OpenAIEmbeddings(
    model="text-embedding-3-small",
    api_key=os.getenv("OPENROUTER_API_KEY"),
    base_url=os.getenv("OPENROUTER_API_BASE", "https://openrouter.ai/api/v1"),
)

# 컬렉션 이름 설정
collection_name = "ppt_multimodal_rag"
try:
    client.create_collection(
        collection_name=collection_name,
        vectors_config={"dense": VectorParams(size=1536, distance=Distance.COSINE)},
    )
    print(f"새 컬렉션 '{collection_name}' 생성됨")
    qdrant = QdrantVectorStore(
        client=client,
        collection_name=collection_name,
        embedding=dense_embeddings,
        retrieval_mode=RetrievalMode.DENSE,
        vector_name="dense",
    )
    qdrant.add_documents(documents)

except Exception as e:
    print(f"에러 발생: {e}")

## Agentic RAG 구현

LangGraph 기반의 Agentic RAG 파이프라인을 구성합니다.
- **의도 분류**: 질문의 복잡도를 판단하여 검색 전략 결정
- **문서 검색**: Qdrant 벡터스토어에서 관련 문서 검색
- **관련성 평가**: 검색된 문서의 질문 관련성 평가
- **답변 생성**: 검색된 문서를 기반으로 최종 답변 생성 (출처 인용 포함)

In [ ]:
from typing import Literal, TypedDict

from langchain_core.documents import Document
from langchain_core.prompts import ChatPromptTemplate
from langgraph.graph import END, StateGraph
from pydantic import BaseModel, Field


# --- State 정의 ---
class AgenticRAGState(TypedDict, total=False):
    query: str
    complexity: Literal["SIMPLE", "COMPLEX"]
    retrieved_docs: list[Document]
    doc_relevant: bool
    answer: str


# --- Retriever 설정 ---
retriever = qdrant.as_retriever(search_kwargs={"k": 5})


# --- 노드 1: 복잡도 분류 ---
class ComplexityResult(BaseModel):
    complexity: Literal["SIMPLE", "COMPLEX"] = Field(description="질문 복잡도")
    reasoning: str = Field(description="판단 근거")


def classify_complexity(state: AgenticRAGState) -> dict:
    structured_llm = llm.with_structured_output(ComplexityResult)
    result = structured_llm.invoke(
        f"""질문의 복잡도를 분류하세요:
- SIMPLE: 단일 슬라이드에서 직접 답변 가능
- COMPLEX: 여러 슬라이드를 종합해야 답변 가능

질문: {state['query']}"""
    )
    print(f"[복잡도] {result.complexity}: {result.reasoning}")
    return {"complexity": result.complexity}


# --- 노드 2: 문서 검색 ---
def retrieve_documents(state: AgenticRAGState) -> dict:
    k = 3 if state.get("complexity") == "SIMPLE" else 6
    docs = retriever.invoke(state["query"])
    docs = docs[:k]
    print(f"[검색] {len(docs)}개 문서 검색 (k={k})")
    for i, doc in enumerate(docs):
        src = doc.metadata.get('file_name', '?')
        page = doc.metadata.get('page', '?')
        print(f"  [{i+1}] {src} p.{page} ({len(doc.page_content)}자)")
    return {"retrieved_docs": docs}


# --- 노드 3: 관련성 평가 ---
class RelevanceGrade(BaseModel):
    relevant: bool = Field(description="문서가 질문에 관련되는지 여부")
    reasoning: str = Field(description="판단 근거")


def grade_documents(state: AgenticRAGState) -> dict:
    docs = state.get("retrieved_docs", [])
    if not docs:
        return {"doc_relevant": False}
    context = "\n---\n".join(d.page_content[:300] for d in docs[:3])
    structured_llm = llm.with_structured_output(RelevanceGrade)
    result = structured_llm.invoke(
        f"질문: {state['query']}\n\n문서 발췌:\n{context}\n\n이 문서들이 질문에 답하는 데 관련이 있습니까?"
    )
    print(f"[관련성] {'관련 있음' if result.relevant else '관련 없음'}: {result.reasoning}")
    return {"doc_relevant": result.relevant}


# --- 노드 4: 답변 생성 ---
GENERATION_PROMPT = ChatPromptTemplate.from_messages([
    ("system", """당신은 PPT 자료를 기반으로 정확히 답변하는 AI입니다.

규칙:
1. 반드시 주어진 컨텍스트만을 기반으로 답변하세요.
2. 각 주장에 출처를 표기하세요: 【파일명, p.페이지번호】
3. 컨텍스트에 없는 내용은 "제공된 자료만으로는 답변하기 어렵습니다"라고 답하세요."""),
    ("human", "컨텍스트:\n{context}\n\n질문: {query}")
])


def generate_answer(state: AgenticRAGState) -> dict:
    docs = state.get("retrieved_docs", [])
    if not docs or not state.get("doc_relevant", False):
        return {"answer": "관련 문서를 찾을 수 없어 답변이 어렵습니다. 질문을 다시 구성해주세요."}
    
    context = "\n---\n".join(
        f"【{d.metadata.get('file_name','?')}, p.{d.metadata.get('page','?')}】\n{d.page_content}"
        for d in docs
    )
    response = llm.invoke(GENERATION_PROMPT.format(context=context, query=state["query"]))
    print(f"[답변] {len(response.content)}자 생성")
    return {"answer": response.content}


# --- LangGraph 그래프 구성 ---
graph = StateGraph(AgenticRAGState)
graph.add_node("classify", classify_complexity)
graph.add_node("retrieve", retrieve_documents)
graph.add_node("grade", grade_documents)
graph.add_node("generate", generate_answer)

graph.set_entry_point("classify")
graph.add_edge("classify", "retrieve")
graph.add_edge("retrieve", "grade")
graph.add_edge("grade", "generate")
graph.add_edge("generate", END)

app = graph.compile()
print("Agentic RAG 그래프 컴파일 완료")
app

### 테스트

PPT 자료에 대한 다양한 질문으로 Agentic RAG를 테스트합니다.

In [ ]:
# 테스트 질문
test_queries = [
    "이 자료의 핵심 주제는 무엇인가요?",
    "주요 데이터나 통계 수치를 요약해주세요.",
    "결론과 시사점을 설명해주세요.",
]

for query in test_queries:
    print(f"\n{'='*60}")
    print(f"질문: {query}")
    print(f"{'='*60}")
    result = app.invoke({"query": query})
    print(f"\n답변:\n{result['answer']}")